# Dynamic Programming — Policy Iteration & Value Iteration Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: build the GridWorld MDP model

Use the same 4×4 GridWorld from Lesson 01. We add a stochastic variant: with probability `0.1` the agent slips to a random perpendicular direction.

In [ ]:
```python

SLIP = 0.1

def transitions(state, action):

    if state == TERMINAL:

        return [(state, 0.0, 1.0)]

    outcomes = []

    for direction, prob in action_probs(action):

        outcomes.append((apply_move(state, direction), -1.0, prob))

    return outcomes

In [ ]:
```

`transitions(s, a)` returns a list of `(s', r, p)`. This is the entire model.

### Step 2: policy evaluation

Given a policy `π(s) = {action: prob}`, iterate the Bellman equation until `V` stops moving:

In [ ]:
```python

def policy_evaluation(policy, gamma=0.99, tol=1e-6):

    V = {s: 0.0 for s in states()}

    while True:

        delta = 0.0

        for s in states():

            v = sum(pi_a * sum(p * (r + gamma * V[s_prime])

                              for s_prime, r, p in transitions(s, a))

                   for a, pi_a in policy(s).items())

            delta = max(delta, abs(v - V[s]))

            V[s] = v

        if delta < tol:

            return V

In [ ]:
```

### Step 3: policy improvement

Replace `π` with the greedy policy w.r.t. `V`. If `π` did not change, return — we are at the optimum.

In [ ]:
```python

def policy_improvement(V, gamma=0.99):

    new_policy = {}

    for s in states():

        best_a = max(

            ACTIONS,

            key=lambda a: sum(p * (r + gamma * V[s_prime])

                              for s_prime, r, p in transitions(s, a)),

        )

        new_policy[s] = best_a

    return new_policy

In [ ]:
```

### Step 4: stitch them together

In [ ]:
```python

def policy_iteration(gamma=0.99):

    policy = {s: "up" for s in states()}   # arbitrary start

    for _ in range(100):

        V = policy_evaluation(lambda s: {policy[s]: 1.0}, gamma)

        new_policy = policy_improvement(V, gamma)

        if new_policy == policy:

            return V, policy

        policy = new_policy

In [ ]:
```

Typical convergence on 4×4: 4–6 outer iterations. Outputs `V*(0,0) ≈ -6` and a policy that strictly decreases the step count.

### Step 5: value iteration (the one-loop version)

In [ ]:
```python

def value_iteration(gamma=0.99, tol=1e-6):

    V = {s: 0.0 for s in states()}

    while True:

        delta = 0.0

        for s in states():

            v = max(sum(p * (r + gamma * V[s_prime])

                       for s_prime, r, p in transitions(s, a))

                   for a in ACTIONS)

            delta = max(delta, abs(v - V[s]))

            V[s] = v

        if delta < tol:

            break

    policy = policy_improvement(V, gamma)

    return V, policy

In [ ]:
```

Same fixed point, fewer lines of code.

## Exercises

In [ ]:
1. **Easy.** Run value iteration on the 4×4 GridWorld with `γ ∈ {0.9, 0.99}`. How many sweeps until `max |ΔV| < 1e-6`? Print `V*` as a 4×4 grid.
2. **Medium.** Compare policy iteration vs value iteration on the *stochastic* GridWorld (slip probability `0.1`). Count: sweeps, wall-clock time, final `V*(0,0)`. Which converges faster in iterations? In wall-clock?
3. **Hard.** Build modified policy iteration: in the evaluation step, run only `k` sweeps instead of to convergence. Plot `V*(0,0)` error vs `k` for `k ∈ {1, 2, 5, 10, 50}`. What does the curve tell you about the evaluation/improvement tradeoff?